In [9]:
import numpy as np
import pandas as pd

In [10]:
import os
import json
dataset_path = "/home/pradeep/Desktop/all_json"

In [11]:
import os


files = os.listdir(dataset_path)

print("Total files:", len(files))
print(files[:10])

Total files: 21577
['1410512.json', '464531.json', '1170459.json', '64838.json', '913445.json', '1227507.json', '65239.json', '1305500.json', '760783.json', '1503565.json']


In [12]:
selected_files = files[:600]

print(len(selected_files))

600


In [13]:
import numpy as np
import pandas as pd
import os
import json
import gc





# -------------------------
# Load T20 Matches
# -------------------------

selected_files = os.listdir(dataset_path)[:500]

columns = [
    "match_id","team","over","ball",
    "batter","bowler",
    "runs","extras","total_runs","wicket"
]

dfs_t20 = []

for file in selected_files:

    with open(os.path.join(dataset_path, file)) as f:
        match = json.load(f)

    match_type = match.get("info", {}).get("match_type","").lower()
    overs = match.get("info", {}).get("overs",0)

    is_t20 = (match_type == "T20") or (overs == 20)

    if not is_t20:
        continue

    match_id = file.split(".")[0]

    rows = []

    for inning in match.get("innings",[]):

        team = inning.get("team")

        for over_data in inning.get("overs",[]):

            over = over_data.get("over")

            for ball_idx, delivery in enumerate(over_data.get("deliveries",[]), start=1):

                rows.append([
                    match_id,
                    team,
                    over,
                    ball_idx,
                    delivery.get("batter"),
                    delivery.get("bowler"),
                    delivery.get("runs",{}).get("batter",0),
                    delivery.get("runs",{}).get("extras",0),
                    delivery.get("runs",{}).get("total",0),
                    1 if "wickets" in delivery else 0
                ])

    if rows:
        dfs_t20.append(pd.DataFrame(rows, columns=columns))

df_t20 = pd.concat(dfs_t20, ignore_index=True)

del dfs_t20
gc.collect()

print("Dataset shape:", df_t20.shape)


# -------------------------
# Feature Engineering
# -------------------------

df_t20[["over","ball","runs","total_runs","wicket"]] = \
df_t20[["over","ball","runs","total_runs","wicket"]].apply(
    pd.to_numeric, downcast="unsigned"
)

df_t20["ball_number"] = df_t20["over"] * 6 + df_t20["ball"]

df_t20["current_score"] = df_t20.groupby(["match_id","team"])["total_runs"].cumsum()

df_t20["current_wickets"] = df_t20.groupby(["match_id","team"])["wicket"].cumsum()

df_t20["wickets_remaining"] = 10 - df_t20["current_wickets"]

df_t20["run_rate"] = df_t20["current_score"] / (df_t20["ball_number"]/6 + 0.1)


# -------------------------
# Phase Feature
# -------------------------

df_t20["phase"] = pd.cut(
    df_t20["over"],
    bins=[-1,5,14,20],
    labels=["powerplay","middle","death"],
    include_lowest=True
)

df_t20["phase"] = df_t20["phase"].astype(str)


# -------------------------
# Player Performance
# -------------------------

df_t20["batter_sr"] = df_t20.groupby("batter")["runs"].transform("mean")

df_t20["bowler_econ"] = df_t20.groupby("bowler")["total_runs"].transform("mean")


df_t20.fillna(0, inplace=True)


# -------------------------
# Encoding
# -------------------------

from sklearn.preprocessing import LabelEncoder

for col in ["team","batter","bowler","phase"]:
    
    le = LabelEncoder()
    
    df_t20[col] = le.fit_transform(df_t20[col].astype(str))


# -------------------------
# Model Dataset
# -------------------------

features = [
    "team",
    "over",
    "ball",
    "batter",
    "bowler",
    "ball_number",
    "current_score",
    "current_wickets",
    "wickets_remaining",
    "run_rate",
    "phase",
    "batter_sr",
    "bowler_econ"
]

X = df_t20[features]

y = df_t20["runs"]   # target should be batter runs


# -------------------------
# Train Test Split
# -------------------------

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)


# -------------------------
# Model Training
# -------------------------

import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.1,
    tree_method="hist",
    objective="multi:softmax",
    num_class=len(np.unique(y))
)

print("Training model...")

model.fit(X_train,y_train)

print("Training Complete")


# -------------------------
# Evaluation
# -------------------------

from sklearn.metrics import accuracy_score

preds = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test,preds))

Dataset shape: (70757, 10)
Training model...
Training Complete
Accuracy: 0.48346523459581686


In [14]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score

# 1. OPTIMIZED DATA LOADING & CLEANING
# Assuming 'df' is your extracted dataframe from the JSONs
def preprocess_cricket_data(df):
    # Ensure numerical types to save memory
    df['over'] = df['over'].astype(int)
    df['ball'] = df['ball'].astype(int)
    
    # Feature Engineering (Optimized)
    df["ball_number"] = df["over"] * 6 + df["ball"]
    df["current_score"] = df.groupby(["match_id", "team"])["total_runs"].cumsum()
    df["current_wickets"] = df.groupby(["match_id", "team"])["wicket"].cumsum()
    df["wickets_remaining"] = 10 - df["current_wickets"]
    df["run_rate"] = df["current_score"] / (df["ball_number"] / 6 + 0.1)
    
    # Advanced: Rolling performance (Last 10 balls for batter)
    # This captures "form" better than a global average
    df['recent_batter_runs'] = df.groupby(['match_id', 'batter'])['runs'].transform(lambda x: x.rolling(window=10, min_periods=1).mean())

    # Phase Encoding
    df['phase'] = pd.cut(df['over'], bins=[-1, 6, 15, 20], labels=['powerplay', 'middle', 'death'])
    
    return df

df = preprocess_cricket_data(df_t20)

# 2. ENCODING (The fast way)
categorical_cols = ['team', 'batter', 'bowler', 'phase']
le_dict = {}

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

# 3. PREPARE FEATURES AND TARGET
# We drop columns that are unavailable at prediction time or redundant
features = [
    'team', 'over', 'ball', 'batter', 'bowler', 'ball_number', 
    'current_score', 'current_wickets', 'run_rate', 
    'wickets_remaining', 'phase', 'recent_batter_runs'
]
X = df[features]
y = df['runs']

# Map target runs to indices (0,1,2,3,4,5,6) -> XGBoost needs 0-indexed classes
# Note: 5 runs is rare but exists; we map categories to unique integers
label_map = {val: i for i, val in enumerate(sorted(y.unique()))}
inv_label_map = {i: val for val, i in label_map.items()}
y_encoded = y.map(label_map)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# 4. FAST TRAINING WITH XGBOOST
# This is much faster than RandomForest for large datasets
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist', # Uses histogram binning for speed
    device="cpu",       # Change to "cuda" if you have a GPU
    objective='multi:softprob',
    n_jobs=-1           # Uses all CPU cores
)

print("Starting training... this should take less than 2 minutes.")
model.fit(X_train, y_train)
print("Training complete!")

# 5. EVALUATION
y_pred = model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
# Use the original run labels for the report
print(classification_report(y_test, y_pred))

Starting training... this should take less than 2 minutes.
Training complete!
Accuracy: 0.5273

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.61      0.61      6146
           1       0.46      0.67      0.55      5044
           2       0.37      0.02      0.03       895
           3       0.00      0.00      0.00        52
           4       0.43      0.17      0.24      1453
           5       0.00      0.00      0.00         1
           6       0.40      0.11      0.17       561

    accuracy                           0.53     14152
   macro avg       0.33      0.22      0.23     14152
weighted avg       0.52      0.53      0.50     14152



/home/pradeep/Desktop/coding/machine learning/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/pradeep/Desktop/coding/machine learning/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/pradeep/Desktop/coding/machine learning/myenv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this b

In [15]:
model.save_model("cricket_model.json")